<a href="https://colab.research.google.com/github/1989v/colab/blob/main/qwen3_embedding_4b_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen3-Embedding-4B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen3-Embedding-4B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# 4B 모델
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-4B",
    model_kwargs={"torch_dtype": torch.float16},
)
#
# 8B 모델
# !pip install -q -U bitsandbytes
# from transformers import BitsAndBytesConfig
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-8B",
#     model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_8bit=True)},
# )

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# 검색어에 대해 상품과의 유사도 비교
import pandas as pd

query = "판문점"

titles = [
    "[서울] DMZ 제3터널 + 도라전망대 + 감악산 현수교",
    "탈북민과 외국인 친구들이 함께하는 특별한 DMZ 투어, 북한의 진짜 이야기🪖",
    "7월한정 [최대11인소그룹] KPOP스타도함께떠나본, 시드니블루마운틴 선셋별보기시크릿투어",
    "엠디 나와 라바트 내부자 투어",
]
enriched = [
    "[서울] DMZ 제3터널 + 도라전망대 + 감악산 현수교 | 파주 | 투어·티켓",
    "탈북민과 외국인 친구들이 함께하는 특별한 DMZ 투어, 북한의 진짜 이야기🪖 | 파주 | 투어·티켓",
    "7월한정 [최대11인소그룹] KPOP스타도함께떠나본, 시드니블루마운틴 선셋별보기시크릿투어 | 시드니 | 투어·티켓",
    "엠디 나와 라바트 내부자 투어 | 라바트 | 투어·티켓",
]

q_emb = model.encode([query], prompt_name="query")

def score(docs):
    d_emb = model.encode(docs)
    return model.similarity(q_emb, d_emb)[0].float().cpu().numpy()

df = pd.DataFrame({
    "상품": [t[:18] for t in titles],
    "타이틀만": score(titles),
    "타이틀+도시+카테고리": score(enriched),
})
df["변화"] = df["타이틀+도시+카테고리"] - df["타이틀만"]
df = df.sort_values("타이틀+도시+카테고리", ascending=False).round(3)
display(df)

,상품,타이틀만,타이틀+도시+카테고리,변화
0,[서울] DMZ 제3터널 + 도라,0.365,0.378,0.014
1,탈북민과 외국인 친구들이 함께하는,0.295,0.355,0.060
3,엠디 나와 라바트 내부자 투어,0.266,0.243,-0.023
2,7월한정 [최대11인소그룹] KP,0.204,0.215,0.011


In [ ]:
import pandas as pd

query = "판문점"

pool = [
    # (키워드, 지역, 유형, 관련어) — DMZ 연관 (정답군)
    ("DMZ",         "파주 경기도",  "안보관광지", "비무장지대 접경지역 땅굴"),
    ("DMZ 투어",    "파주 경기도",  "투어",       "안보관광 제3터널 도라전망대"),
    ("임진각",      "파주 경기도",  "관광지",     "평화누리공원 안보관광 실향민"),
    ("파주",        "경기도",       "도시",       "출판단지 아울렛 안보관광"),
    ("통일전망대",  "고성 강원도",  "안보관광지", "북한 조망 금강산 전망"),
    ("비무장지대",  "파주 경기도",  "안보관광지", "군사분계선 접경지역"),
    ("공동경비구역","파주 경기도",  "안보관광지", "JSA 군사분계선 회담장"),
    ("JSA",         "파주 경기도",  "안보관광지", "공동경비구역 군사분계선"),
    ("제3땅굴",     "파주 경기도",  "안보관광지", "남침용 땅굴 DMZ 안보관광"),
    # 국내
    ("서울",        "대한민국",     "도시",       "궁궐 쇼핑 한강"),
    ("경복궁",      "서울",         "고궁",       "한복 수문장교대식 광화문"),
    ("남산타워",    "서울",         "전망대",     "N서울타워 케이블카 야경"),
    ("북촌한옥마을","서울",         "관광지",     "한옥 골목 전통문화"),
    ("가평",        "경기도",       "도시",       "쁘띠프랑스 수상레저 펜션"),
    ("남이섬",      "춘천 강원도",  "관광지",     "메타세쿼이아 자전거 겨울연가"),
    ("제주도",      "대한민국",     "섬",         "한라산 올레길 흑돼지"),
    ("부산",        "대한민국",     "도시",       "해운대 광안리 자갈치시장"),
    ("강릉",        "강원도",       "도시",       "경포대 커피거리 바다"),
    ("속초",        "강원도",       "도시",       "설악산 아바이마을 바다"),
    ("경주",        "경상북도",     "도시",       "불국사 첨성대 역사여행"),
    # 해외
    ("오사카",      "일본 간사이",  "도시",       "유니버설스튜디오 도톤보리"),
    ("도쿄",        "일본 간토",    "도시",       "디즈니랜드 시부야 스카이트리"),
    ("교토",        "일본 간사이",  "도시",       "기요미즈데라 아라시야마 기모노"),
    ("후쿠오카",    "일본 규슈",    "도시",       "하카타 텐진 야타이"),
    ("다낭",        "베트남",       "도시",       "바나힐 미케비치 호이안"),
    ("나트랑",      "베트남",       "도시",       "빈펄랜드 머드온천 비치"),
    ("방콕",        "태국",         "도시",       "왓포 카오산로드 짜뚜짝"),
    ("세부",        "필리핀",       "섬",         "호핑투어 스노클링 막탄"),
    ("발리",        "인도네시아",   "섬",         "우붓 스미냑 요가"),
    ("괌",          "미국령",       "섬",         "투몬비치 면세쇼핑 가족여행"),
    ("파리",        "프랑스",       "도시",       "에펠탑 루브르 몽마르트"),
    # 테마
    ("스노클링",    "동남아 휴양지","액티비티",   "호핑투어 바다 산호"),
    ("온천",        "일본 국내",    "테마",       "료칸 노천탕 힐링"),
    ("골프여행",    "동남아 일본",  "테마",       "라운딩 캐디 리조트"),
    ("벚꽃놀이",    "봄 시즌",      "테마",       "벚꽃명소 피크닉 사진"),
    ("야경투어",    "도시 여행",    "테마",       "전망대 크루즈 나이트투어"),
]

bare = [k for k, *_ in pool]
docs = [f"{k} | {r} | {t} | {w}" for k, r, t, w in pool]

df = pd.DataFrame({
    "keyword":      bare,
    "맨몸_대칭":     model.similarity(model.encode([query]), model.encode(bare))[0].float().cpu().numpy(),
    "태깅_비대칭":   model.similarity(model.encode([query], prompt_name="query"), model.encode(docs))[0].float().cpu().numpy(),
})
df = df.sort_values("태깅_비대칭", ascending=False).reset_index(drop=True)
df.index += 1
display(df.round(3).head(15))

,keyword,맨몸_대칭,태깅_비대칭
1,공동경비구역,0.868,0.435
2,JSA,0.627,0.429
3,비무장지대,0.856,0.395
4,가평,0.854,0.373
5,통일전망대,0.869,0.368
6,경복궁,0.828,0.364
7,DMZ,0.733,0.353
8,DMZ 투어,0.697,0.348
9,부산,0.835,0.335
10,파주,0.862,0.317
